<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.2-power-grid-stability-prediction/Ex12.2_04_screening.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.2 · Notebook 04 — Screening

**Paired with L12.2 · Prediction of Power Grid Stability**

You have two models that predict a critical clearing time. Neither of them is
the product. The product is a **decision**: given the present dispatch and a
list of contingencies, which ones does an engineer look at before lunch?

That decision has a threshold in it, and choosing the threshold is not a
modelling question. It is a question about what the two mistakes cost.

> A missed insecure contingency is a blackout the screen did not see.
> A false alarm is an engineer's afternoon.

Those are not commensurable and no metric that averages them is honest. This
notebook makes the asymmetry explicit, finds the threshold that removes the
first kind of error entirely, and prices what that costs in the second.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.2-power-grid-stability-prediction/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
import os
os.makedirs(pb.RESULTS, exist_ok=True)

data = pb.build_dataset(n_ops=180, seed=12)
cases = pb.contingencies()
cct, op_id, cont_id = data["cct"], data["op_id"], data["cont_id"]

nb02 = pb.load("nb02_dense")
nb03 = pb.load("nb03_graph")
test = nb02["test"].astype(bool)

y_true = cct[test]
op_te = op_id[test]
cont_te = cont_id[test]

predictions = {
    "linear": nb02["lin_test"],
    "dense": nb02["pred_test"],
    "graph": nb03["pred_test"],
}

print(f"  {test.sum()} held-out cases from "
      f"{len(np.unique(op_te))} dispatches")
print(f"  truly insecure at {pb.PROTECTION_TIME*1e3:.0f} ms: "
      f"{int((y_true <= pb.PROTECTION_TIME).sum())}")
for name, p in predictions.items():
    print(f"  {name:<8s} test MAE {np.abs(y_true-p).mean()*1e3:6.2f} ms")

**What you should see.** 216 cases from 36 dispatches, **26 of them truly
insecure**, and three MAE figures. The linear one is **20.72 ms**; the other two
are yours.

---

## 1 · One dispatch, ranked

Start with what a screen actually produces: a list, ordered.

In [ ]:
which = int(np.unique(op_te)[0])
sel = op_te == which
labels = [cases[k]["label"].split("(")[0].strip() for k in cont_te[sel]]

print(f"  dispatch {which}:"
      f"  total load {-data['ops'][which,0,[2,3,4]].sum():.3f} p.u.,"
      f"  Gen east {data['ops'][which,0,1]:.3f} p.u.")
print()
print(f"  {'contingency':<26s}{'true':>10s}{'linear':>10s}{'dense':>10s}{'graph':>10s}")
order = np.argsort(predictions["graph"][sel])
for j in order:
    row = f"  {labels[j]:<26s}{y_true[sel][j]*1e3:>10.1f}"
    for name in ("linear", "dense", "graph"):
        row += f"{predictions[name][sel][j]*1e3:>10.1f}"
    print(row + ("   <- INSECURE" if y_true[sel][j] <= pb.PROTECTION_TIME else ""))

pb.plot_screening(y_true[sel], predictions["graph"][sel], labels=labels,
                  title=f"dispatch {which}: contingencies ranked by predicted CCT")
plt.tight_layout(); plt.show()

**What you should see.** Six rows, ordered by the graph model's prediction, with
the true clearing time beside each. Whether the ordering is right matters more
here than whether the numbers are: a screen that gets every value wrong by 30 ms
but ranks correctly is still useful, and one that is accurate on average but
puts the dangerous case fourth is not.

---

## 2 · The threshold sweep

Screening at exactly `PROTECTION_TIME` says: flag a contingency when the model
thinks it is insecure. That is the obvious rule and it is the wrong one,
because the model has error and the error is symmetric while the consequences
are not.

Raising the screening threshold above the protection time buys margin. Every
millisecond of margin catches contingencies the model under-estimated, and
costs contingencies it over-estimated.

### Your turn

In [ ]:
# TODO: sweep the screening threshold for each model and count both errors.
#
#   THRESHOLDS = np.arange(0.10, 0.31, 0.005)
#
#   For each model and each threshold:
#       flagged      = prediction <= threshold
#       true_insecure = y_true <= pb.PROTECTION_TIME       # NOTE: fixed
#       missed = (true_insecure & ~flagged).sum()
#       alarms = (~true_insecure & flagged).sum()
#
#   The threshold moves; the DEFINITION of insecure does not. Insecure means
#   the fault outlasts the breakers, which is a fact about the power system.
#   The threshold is a fact about how much you distrust your model.
#
#   Put the results in sweep = {name: {"missed": [...], "alarms": [...]}},
#   aligned with THRESHOLDS, and record for each model
#       zero_miss[name] -- the SMALLEST threshold at which missed == 0
#       cost[name]      -- the number of false alarms at that threshold
#
#   If a model never reaches zero misses inside the sweep, record np.nan and
#   say so in your report. That is a result, not a bug.

raise NotImplementedError("Sweep the screening threshold for all three models")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
pb.use_course_style()
for i, (name, s) in enumerate(sweep.items()):
    ax[0].plot(np.asarray(THRESHOLDS)*1e3, s["missed"], "-", lw=1.8,
               color=CYCLE[i], label=name)
    ax[1].plot(np.asarray(THRESHOLDS)*1e3, s["alarms"], "-", lw=1.8,
               color=CYCLE[i], label=name)
for a, t in zip(ax, ["missed insecure contingencies", "false alarms"]):
    a.axvline(pb.PROTECTION_TIME*1e3, color=pb.ORANGE, lw=1.4, ls="--",
              label="protection time")
    a.set_xlabel("screening threshold  (ms)"); a.set_ylabel("cases")
    a.set_title(t, fontsize=10); a.legend(fontsize=8)
ax[0].axhline(0, color=pb.GREEN, lw=1.0, ls=":")
fig.tight_layout(); plt.show()

rows = [[name,
         "—" if not np.isfinite(zero_miss[name]) else f"{zero_miss[name]*1e3:.0f}",
         "—" if not np.isfinite(zero_miss[name]) else f"{cost[name]}",
         f"{pb.confusion(y_true, predictions[name])['missed']}",
         f"{pb.confusion(y_true, predictions[name])['alarms']}"]
        for name in sweep]
print(error_table(rows, ["model", "zero-miss threshold [ms]",
                         "false alarms there",
                         f"missed at {pb.PROTECTION_TIME*1e3:.0f} ms",
                         f"alarms at {pb.PROTECTION_TIME*1e3:.0f} ms"]))
print()
print(f"  total test cases {len(y_true)}, truly insecure "
      f"{int((y_true <= pb.PROTECTION_TIME).sum())}")

**What you should see.** Two curves per model. The missed-cases curve falls to
zero as the threshold rises and then stays there; the false-alarm curve climbs
without limit. The question is where they cross your tolerance, and the answer
is not symmetric.

For the **linear baseline**, which you can check exactly because it involves no
training, the sweep gives:

```
   thr[ms]   missed   alarms  flagged
       100       18        0        8
       120        4        0       22
       130        2        0       24
       140        0        5       31
       160        0       15       41
       180        0       26       52
       200        0       37       63
       250        0       90      116
```

with a **smallest zero-miss threshold of 138 ms** — three false alarms, 29 of
216 contingencies flagged.

Read that table as an engineer rather than as a modeller. Screening at 130 ms
looks attractive: no false alarms at all, only 24 cases to look at. It also
misses **two contingencies that would take the system down**. Screening at 200
ms misses nothing and hands somebody 63 cases instead of 26 — 37 of them
pointless. Thirty-seven wasted studies against two blackouts is not a close
call, and no accuracy figure in this exercise set would have told you that.

The asymmetry has a name in this industry. A screening tool is allowed to be
**conservative** and is not allowed to be **optimistic**, and the two are not
opposite ends of one axis: a conservative screen costs money, an optimistic one
costs load.

---

## 3 · What the threshold does not fix

A threshold high enough to catch every insecure contingency in *this* test split
is not a guarantee about the next one. Two things it cannot do anything about:

**Systematic bias.** Notebook 02 section 5 showed a dense model over-predicting
by **+67 ms** on a topology it had never seen. A screening margin of 60 ms would
have absorbed almost none of that, because the error is not noise around the
right answer; it is the wrong answer.

**Censored labels.** 17 % of the training labels are the number 500, which means
"we stopped looking". A model that has learned to output 500 confidently is
confident about a stopping rule.

The honest use of a screen is therefore narrower than it first appears, and it
is worth writing down before section 4 measures the saving:

> The surrogate does not replace the simulator. It replaces the **exhaustive
> sweep**. Everything it flags gets simulated properly, and the simulator's
> answer is the one that counts.

---

## 4 · What the screen is actually worth

If everything flagged gets simulated anyway, the saving is the sweep it
replaces. Price it.

### Your turn

In [ ]:
# TODO: measure the two things a control room would ask for.
#
#   (a) RANKING. For each held-out dispatch, does the model put the genuinely
#       worst contingency first?
#
#           for i in np.unique(op_te):
#               m = op_te == i
#               true_worst = np.argmin(y_true[m])
#               ranked     = np.argsort(prediction[m])
#               ... count ranked[0] == true_worst, and true_worst in ranked[:2]
#
#       Record rank1[name] and rank2[name] as fractions of the 36 dispatches.
#
#   (b) THE TOP-k POLICY. Simulate only the k lowest-predicted contingencies
#       per dispatch and count how many truly insecure cases you catch.
#
#           for k in (1, 2, 3, 6):
#               caught[k] = number of insecure cases inside the top k
#
#       Record caught[name][k] and the work done, k / pb.N_CONTINGENCY.
#
#   Do both for every model in `predictions`.
#
# Predict the answer to (b) before you run it. If a model ranks the worst
# contingency first for every dispatch, what fraction of insecure cases does
# k = 1 catch? It is not 100 %, and the reason is worth a sentence in your
# report.

raise NotImplementedError("Measure ranking quality and the top-k policy")

In [ ]:
n_ops_te = len(np.unique(op_te))
n_insecure = int((y_true <= pb.PROTECTION_TIME).sum())

rows = []
for name in predictions:
    rows.append([name, f"{rank1[name]*100:.0f} %", f"{rank2[name]*100:.0f} %"]
                + [f"{caught[name][k]}/{n_insecure}" for k in (1, 2, 3)])
print(error_table(rows, ["model", "worst ranked 1st", "worst in top 2",
                         "caught, k=1", "k=2", "k=3"]))

print()
print(f"  work done, k=1: {100/pb.N_CONTINGENCY:.0f} % of the sweep")
print(f"  work done, k=2: {200/pb.N_CONTINGENCY:.0f} %")
print(f"  work done, k=3: {300/pb.N_CONTINGENCY:.0f} %")
print()
label_seconds = float(data["seconds"]) / len(cct) if "seconds" in data else 0.31
print(f"  one label costs about {label_seconds*1e3:.0f} ms of simulation")
print(f"  full sweep over the test split : "
      f"{len(y_true)*label_seconds:.1f} s")
print(f"  top-2 policy                   : "
      f"{2*n_ops_te*label_seconds:.1f} s of simulation"
      f" + one forward pass per case")

**What you should see.** For the linear baseline the ranking is perfect —
**the truly worst contingency is ranked first for all 36 dispatches** — and yet
the top-k table reads:

```
  k=1 : simulate 17 % of the sweep, catch 11 of 26 insecure (42 %)
  k=2 : simulate 33 %,              catch 16 of 26 (62 %)
  k=3 : simulate 50 %,              catch 21 of 26 (81 %)
```

**Perfect ranking, and 58 % of the insecure cases missed at k = 1.** That is
worth sitting with, because it is the trap the "rank and take the top one"
framing walks straight into.

The reason is that a dispatch can have **more than one** insecure contingency.
On a heavily loaded evening the tie outage is insecure, and so is the base case,
and so is the line-3 outage. Ranking tells you which is *worst*; it does not
tell you how many are *bad*. A fixed-k policy answers the wrong question.

The threshold policy of section 2 answers the right one: it flags every case
whose predicted margin is thin, however many that turns out to be — 8 on a quiet
night, 40 on a cold evening. The work is variable, which is inconvenient, and
that is the price of asking the question that matters.

Report which policy your models support and at what cost. If your graph model
and your dense model differ here more than they differed on MAE, say so — that
is the kind of difference that survives contact with an operating room.

---

## 5 · Save

In [ ]:
pb.save("nb04_screening",
        thresholds=np.asarray(THRESHOLDS),
        missed_linear=np.asarray(sweep["linear"]["missed"]),
        alarms_linear=np.asarray(sweep["linear"]["alarms"]),
        missed_dense=np.asarray(sweep["dense"]["missed"]),
        alarms_dense=np.asarray(sweep["dense"]["alarms"]),
        missed_graph=np.asarray(sweep["graph"]["missed"]),
        alarms_graph=np.asarray(sweep["graph"]["alarms"]),
        zero_miss=np.array([zero_miss[n] for n in predictions]),
        zero_miss_cost=np.array([cost[n] for n in predictions]),
        rank1=np.array([rank1[n] for n in predictions]),
        model_names=np.array(list(predictions)))

---

## 6 · Before you move on

1. Give your zero-miss threshold for each model, in milliseconds, and the
   number of false alarms it costs. Then say which model you would deploy and
   why — the answer is allowed to be the less accurate one.
2. Screening at 130 ms would have produced no false alarms at all on the linear
   model's predictions. Write the sentence you would say to a control-room
   manager who proposes it.
3. The top-1 policy catches 42 % of insecure cases despite ranking the worst one
   correctly every time. Explain the mechanism in one sentence, and name the
   property of the *dispatch* that makes it happen.
4. Section 3 claimed a screening margin cannot absorb a systematic bias. Give
   the check you would run, before deployment, that would have detected the
   +67 ms bias of notebook 02 section 5.

Next: **notebook 05**, the report.